### 馬達論文研究
### 第四步 辨識未知資料是否為新資料
### 馬達A 6000RPM
### 使用CNN_Res
### 先使用HDBSCAN分群再使用馬氏距離判定未知故障

In [12]:
import os
import random
import warnings
from collections import Counter, defaultdict

import hdbscan
import numpy as np
import pandas as pd
from scipy.spatial.distance import cdist
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from tensorflow.keras.layers import Flatten
from tensorflow.keras.models import Model, load_model

warnings.filterwarnings("ignore")

In [13]:
# --------------------------- 全域參數 ---------------------------
ROOT_DIR = os.getcwd()
FEATURE_DIR = os.path.join(ROOT_DIR, "階段3", "myfeature")
MODEL_PATH = os.path.join(ROOT_DIR, "階段3", "model", "ResNet_A6000_1.keras")
RPM = "6000rpm"            # 目前只跑 A 6000 RPM，可自行改
T_CODE = "T3"              # 資料夾代號 (時間點)
LABEL_ORDER = [
    "8screws",
    "1screws",
    "2screws",
    "3screws",
    "4screws",
]  # 舊資料標籤順序

In [14]:
# ➜ 想一次測多種設定就寫在這裡
MIN_CLUSTER_SIZES = list(range(22,25)) # 最小群集大小
CONF_LEVELS = [0.90, 0.95, 0.99]
UNKNOWN_BATCHES = [
    ["5screws"],
    ["5screws", "6screws"],
    ["5screws", "6screws", "7screws"],
    ["5screws", "6screws", "7screws", "3_14screws"],
    ["5screws", "6screws", "7screws", "3_14screws", "4_146screws"],
]

PER_SCREW_LIMIT   = 60      # 每種未知 screws 只取前 60 筆
MAX_COMBO_SAMPLE  = 450      # 混合資料後最多抽多少筆避免過大

In [15]:
def load_and_concat(base_dir: str, t_code: str, rpm: str, screws_list):
    """讀取多個 screws 資料並 concat"""
    dfs = []
    for screws in screws_list:
        path = os.path.join(
            base_dir,
            f"{t_code}",
            rpm,
            screws,
            f"{t_code}_Group_feature_data_clean.csv",
        )
        if os.path.exists(path):
            df = pd.read_csv(path)
            df["screws"] = screws
            dfs.append(df)
        else:
            print(f"⚠️ 缺少檔案：{path}")
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

In [16]:
# ---------- 1. 讀取舊資料 ----------
print("\n=== 1. Loading training data ===")
train_df = load_and_concat(FEATURE_DIR, T_CODE, RPM, LABEL_ORDER)
label_map = {v: i for i, v in enumerate(LABEL_ORDER)}
train_df["label"] = train_df["screws"].map(label_map)
X = train_df.drop(["screws", "label"], axis=1).values
y = train_df["label"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = RobustScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f"✔️ Train : {X_train.shape}  |  Test : {X_test.shape}")



=== 1. Loading training data ===
✔️ Train : (1218, 105)  |  Test : (305, 105)


In [17]:
# ---------- 2. CNN 特徵 ----------
cnn = load_model(MODEL_PATH, compile=False)
feat_model = Model(inputs=cnn.input, outputs=Flatten()(cnn.get_layer("max_pooling1d").output))
X_train_f = feat_model.predict(X_train_s.reshape(-1, X_train_s.shape[1], 1), verbose=0)
X_test_f  = feat_model.predict(X_test_s.reshape(-1, X_test_s.shape[1], 1),  verbose=0)

In [18]:
# ---------- 3. 以不同 min_cluster_size 建 HDBSCAN & 清離群 ----------
def fit_hdbscan_inliers(features, labels, mcs):
    cl = hdbscan.HDBSCAN(min_cluster_size=mcs, min_samples=3, cluster_selection_method="eom").fit(features)
    mask = cl.labels_ != -1
    return features[mask], labels[mask]

In [19]:
# ---------- 4. Mahalanobis 工具函式 ----------
def mahalanobis_stats(X, conf):
    mu  = X.mean(axis=0)
    cov = np.cov(X, rowvar=False)
    inv = np.linalg.pinv(cov + np.eye(cov.shape[0]) * 1e-6)
    d   = np.sqrt(((X - mu) @ inv * (X - mu)).sum(axis=1))
    thr = np.percentile(d, conf * 100)
    return mu, inv, thr


In [20]:
# ---------- 5‑A. Test‑set validation ----------
def validate_test_set(features, true_labels, mcs, mu, inv, thr):
    print("\n--- Test‑set validation")
    cl = hdbscan.HDBSCAN(min_cluster_size=mcs, min_samples=3, cluster_selection_method="leaf").fit(features)
    lbl = cl.labels_
    for cid in sorted(set(lbl)):
        idx = np.where(lbl == cid)[0]
        names = [LABEL_ORDER[l] for l in true_labels[idx]]
        cnts = Counter(names)
        tag  = "outliers" if cid == -1 else f"C{cid}"
        info = "，".join(f"{k} {v}筆" for k, v in cnts.items())
        print(f" {tag:<8}: {info}")
    for cid in sorted(set(lbl) - {-1}):
        idx  = np.where(lbl == cid)[0]
        ctr  = features[idx].mean(axis=0, keepdims=True)
        dist = np.sqrt(((ctr - mu) @ inv * (ctr - mu)).sum())
        stat = "New" if dist > thr else "Known"
        print(f"  * C{cid:<2} | size {len(idx):3d} | dist {dist:6.2f} | {stat}")

In [21]:
# ---------- 5-B. Unknown data pipeline ----------
def evaluate_unknown_batch(batch, mcs, mu, inv, thr):
    unknown_df = load_and_concat(FEATURE_DIR, T_CODE, RPM, batch)
    if unknown_df.empty:
        print("❌ 無資料，跳過")
        return
    # 每種 screws 取前 60 筆
    unknown_df = unknown_df.groupby("screws", group_keys=False).head(PER_SCREW_LIMIT)

    # 混入舊 Test set（標示 old_test）方便觀察
    old_df = pd.DataFrame(X_test, columns=train_df.columns.drop(["screws", "label"]))
    old_df["screws"] = "old_test"
    combo = pd.concat([old_df, unknown_df], ignore_index=True)
    combo = combo.sample(min(len(combo), MAX_COMBO_SAMPLE), random_state=42)

    X_new_s = scaler.transform(combo.drop("screws", axis=1).values)
    X_new_f = feat_model.predict(X_new_s.reshape(-1, X_new_s.shape[1], 1), verbose=0)

    cl = hdbscan.HDBSCAN(min_cluster_size=mcs, min_samples=3, cluster_selection_method="leaf").fit(X_new_f)
    lbl = cl.labels_
    cnt = Counter(lbl)

    print(f"\n--- Unknown {batch} | mcs {mcs}")
    for k, v in cnt.items():
        tag = "outliers" if k == -1 else f"C{k}"
        print(f"  {tag:<8}: {v} pts")
    for cid in sorted(set(lbl) - {-1}):
        idx  = np.where(lbl == cid)[0]
        ctr  = X_new_f[idx].mean(axis=0, keepdims=True)
        dist = np.sqrt(((ctr - mu) @ inv * (ctr - mu)).sum())
        stat = "New Fault" if dist > thr else "Known"
        print(f"  C{cid:<2} | size {len(idx):3d} | dist {dist:6.2f} | {stat}")

In [22]:
# -------------------- 主程式迴圈 --------------------
for mcs in MIN_CLUSTER_SIZES:
    X_filt, y_filt = fit_hdbscan_inliers(X_train_f, y_train, mcs)
    for conf in CONF_LEVELS:
        MU, INV, THR = mahalanobis_stats(X_filt, conf)
        print("\n==============================")
        print(f"▶ mcs = {mcs} | conf = {conf} | thr = {THR:.2f}")
        print("==============================")
        validate_test_set(X_test_f, y_test, mcs, MU, INV, THR)
        for batch in UNKNOWN_BATCHES:
            evaluate_unknown_batch(batch, mcs, MU, INV, THR)


▶ mcs = 22 | conf = 0.9 | thr = 30.80

--- Test‑set validation
 outliers: 3screws 36筆，1screws 6筆，4screws 2筆，2screws 2筆
 C0      : 8screws 63筆
 C1      : 1screws 54筆
 C2      : 2screws 54筆
 C3      : 3screws 31筆
 C4      : 4screws 57筆
  * C0  | size  63 | dist   8.22 | Known
  * C1  | size  54 | dist   9.41 | Known
  * C2  | size  54 | dist  10.79 | Known
  * C3  | size  31 | dist  12.00 | Known
  * C4  | size  57 | dist   7.17 | Known

--- Unknown ['5screws'] | mcs 22
  C0      : 62 pts
  outliers: 47 pts
  C1      : 54 pts
  C3      : 60 pts
  C5      : 57 pts
  C2      : 54 pts
  C4      : 31 pts
  C0  | size  62 | dist   8.32 | Known
  C1  | size  54 | dist   9.41 | Known
  C2  | size  54 | dist  10.79 | Known
  C3  | size  60 | dist 102.20 | New Fault
  C4  | size  31 | dist  12.00 | Known
  C5  | size  57 | dist   7.17 | Known

--- Unknown ['5screws', '6screws'] | mcs 22
  C3      : 59 pts
  C1      : 54 pts
  C0      : 62 pts
  C4      : 55 pts
  outliers: 53 pts
  C2      : 54 